# Phase 5 -- Advanced Ensemble and Stacking
## Human vs. AI-Generated Text Classification

**What this notebook does:**

Phase 4 built a basic weighted soft-voting ensemble (DeBERTa + SVC + LR) with pseudo-labelling. This notebook pushes further with **advanced ensembling techniques** that squeeze out the last fraction of F1:

1. **Stacking Meta-Learner** -- Train a small LR on top of base model probabilities (3 x 6 = 18 meta-features)
2. **Rank-Based Ensemble** -- Use rank averaging instead of raw probabilities (robust to miscalibration)
3. **Per-Class Specialist Voting** -- Let each model "own" the classes it is best at
4. **Confidence-Aware Blending** -- Weight models dynamically based on per-sample confidence
5. **Full comparison** with bar charts showing which approach wins

**Inputs from Phase 3 and 4:**
- -- DeBERTa logits, probs, split indices
- -- Ensemble probs, SVC/LR CV probs, config

**Outputs:**
- -- Stacking model, best probs, config
- -- Advanced ensemble submissions
- -- Method comparison chart


In [ ]:
import os, warnings, gc, time, json, re, string
os.chdir('/Users/aliivaezii/Documents/MALTO')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from collections import Counter
from scipy import sparse

import torch
import torch.nn as nn

from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.model_selection import StratifiedKFold

import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
import joblib

SEED = 42
NUM_LABELS = 6
LABEL_MAP = {0: 'Human', 1: 'DeepSeek', 2: 'Grok', 3: 'Claude', 4: 'Gemini', 5: 'ChatGPT'}

np.random.seed(SEED)

train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')
y_all = train_df['LABEL'].values

print(f'Train: {train_df.shape}, Test: {test_df.shape}')
print(f'Labels: {dict(zip(*np.unique(y_all, return_counts=True)))}')

## 1. Load All Artifacts from Phase 3 & 4

We load everything saved by earlier notebooks:
- **DeBERTa** logits and calibrated probabilities (Phase 3)
- **SVC & LR** 5-fold CV probabilities (Phase 4)
- **Ensemble config** with weights and thresholds (Phase 4)
- **Split indices** so we use the same train/val/cal partitions

In [ ]:
# --- Phase 3 artifacts (DeBERTa) ---
deberta_test_logits = np.load('models/transformer_test_logits.npy')
deberta_test_probs = np.load('models/transformer_test_probs_calibrated.npy')
deberta_test_probs_raw = np.load('models/transformer_test_probs_raw.npy')
deberta_val_preds = np.load('models/transformer_val_preds.npy')
deberta_val_logits = np.load('models/transformer_val_logits.npy')
deberta_cal_logits = np.load('models/transformer_cal_logits.npy')

splits = np.load('models/transformer_split_indices.npz')
train_idx = splits['train_idx']
val_idx = splits['val_idx']
cal_idx = splits['cal_idx']

# --- Phase 4 artifacts (SVC/LR CV probs, ensemble config) ---
svc_test_probs_cv = np.load('models/ensemble_svc_test_probs_cv.npy')
lr_test_probs_cv = np.load('models/ensemble_lr_test_probs_cv.npy')
svc_oof_probs = np.load('models/ensemble_svc_oof_probs.npy')
lr_oof_probs = np.load('models/ensemble_lr_oof_probs.npy')

with open('models/ensemble_config.json') as f:
    ensemble_config = json.load(f)

temperature = ensemble_config['temperature']

print('Phase 3 artifacts:')
print(f' test_logits: {deberta_test_logits.shape}')
print(f' test_probs: {deberta_test_probs.shape}')
print(f' val_logits: {deberta_val_logits.shape}')
print(f' cal_logits: {deberta_cal_logits.shape}')
print(f' Split: train={len(train_idx)}, val={len(val_idx)}, cal={len(cal_idx)}')

print('\nPhase 4 artifacts:')
print(f' SVC OOF probs: {svc_oof_probs.shape}')
print(f' LR OOF probs: {lr_oof_probs.shape}')
print(f' SVC test probs: {svc_test_probs_cv.shape}')
print(f' LR test probs: {lr_test_probs_cv.shape}')
print(f' Temperature: {temperature:.2f}')
print(f' Phase 4 weights: {ensemble_config["ensemble_weights"]}')

## 2. Reconstruct DeBERTa OOF Probabilities

DeBERTa was trained on a **single split** (train/val/cal), so we don't have true out-of-fold (OOF) probabilities for the full training set. We do have:
- **val_logits** -- model predictions on val set (unseen during training)
- **cal_logits** -- model predictions on calibration set (unseen during training)

For the **train subset** we have no predictions (the model was trained on these). We'll handle this by:
1. Using val+cal as the "OOF" portion for DeBERTa
2. For stacking, only training the meta-learner on val+cal indices where all models have OOF predictions

In [ ]:
# DeBERTa OOF: we have predictions on val and cal sets
# Convert logits to calibrated probabilities using the temperature found in Phase 3
deberta_val_probs = torch.softmax(
    torch.tensor(deberta_val_logits, dtype=torch.float32) / temperature, dim=-1
    ).numpy()

deberta_cal_probs = torch.softmax(
    torch.tensor(deberta_cal_logits, dtype=torch.float32) / temperature, dim=-1
    ).numpy()

# Build a "partial OOF" array for DeBERTa: only val+cal have values
deberta_oof_probs = np.zeros((len(y_all), NUM_LABELS))
deberta_oof_probs[val_idx] = deberta_val_probs
deberta_oof_probs[cal_idx] = deberta_cal_probs

# Indices where we have OOF predictions from ALL models
# SVC/LR have full OOF (5-fold CV covers all samples)
# DeBERTa has OOF only for val+cal
oof_indices = np.concatenate([val_idx, cal_idx])
oof_indices.sort()

print(f'DeBERTa OOF coverage: {len(oof_indices)}/{len(y_all)} samples ({len(oof_indices)/len(y_all)*100:.0f}%)')
print(f'SVC/LR OOF coverage: {len(y_all)}/{len(y_all)} samples (100%)')
print(f'Stacking meta-learner will train on {len(oof_indices)} samples')

# Quick sanity check: DeBERTa val F1
deberta_val_f1 = f1_score(y_all[val_idx], deberta_val_probs.argmax(axis=1), average='macro')
svc_oof_f1 = f1_score(y_all, svc_oof_probs.argmax(axis=1), average='macro')
lr_oof_f1 = f1_score(y_all, lr_oof_probs.argmax(axis=1), average='macro')

print(f'\nBase model performance:')
print(f' DeBERTa val F1: {deberta_val_f1:.4f}')
print(f' SVC 5-fold OOF: {svc_oof_f1:.4f}')
print(f' LR 5-fold OOF: {lr_oof_f1:.4f}')

## 3. Method A -- Stacking Meta-Learner

**Stacking** is one of the most powerful ensemble techniques. Instead of manually choosing weights, we train a **meta-learner** (a small model) that learns how to optimally combine the base model predictions.

**How it works:**
1. Collect OOF probability predictions from each base model (DeBERTa, SVC, LR)
2. Stack these into a single feature matrix: each sample has 3×6 = 18 features (3 models × 6 class probabilities)
3. Train a Logistic Regression on this meta-feature matrix
4. The meta-learner learns which model to trust for which class

**Why this is better than manual weights:**
- Manual weights apply the same weight to all classes
- Stacking can learn "trust DeBERTa for DeepSeek but trust SVC for Human"

In [ ]:
# Build stacking meta-features from OOF predictions
# Shape: (n_samples, 3 models × 6 classes = 18 features)

# For the meta-learner, use only indices where ALL models have OOF preds
meta_X = np.hstack([
    deberta_oof_probs[oof_indices],
    svc_oof_probs[oof_indices],
    lr_oof_probs[oof_indices],
    ])
meta_y = y_all[oof_indices]

print(f'Meta-feature matrix: {meta_X.shape}')
print(f' Features: DeBERTa(6) + SVC(6) + LR(6) = 18')
print(f' Samples: {len(meta_y)} (val+cal from Phase 3 split)')
print(f' Label distribution: {dict(zip(*np.unique(meta_y, return_counts=True)))}')

# Build corresponding test meta-features
meta_X_test = np.hstack([
    deberta_test_probs, # DeBERTa calibrated
    svc_test_probs_cv, # SVC 5-fold averaged
    lr_test_probs_cv, # LR 5-fold averaged
    ])
print(f'Test meta-features: {meta_X_test.shape}')

### 🏗️ Stacking Meta-Features

Build stacking features from out-of-fold probability predictions. The meta-learner sees calibrated probability vectors from each base model.

In [ ]:
# Train stacking meta-learner with 5-fold CV on the OOF data
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

stack_oof_preds = np.zeros(len(meta_y), dtype=int)
stack_oof_probs = np.zeros((len(meta_y), NUM_LABELS))
stack_test_probs = np.zeros((len(test_df), NUM_LABELS))
stack_fold_scores = []

print('Training stacking meta-learner (LR) with 5-fold CV...')
for fold, (tr_idx, vl_idx) in enumerate(skf.split(meta_X, meta_y)):
    meta_lr = LogisticRegression(
        C=1.0, class_weight='balanced', max_iter=5000,
        solver='lbfgs', random_state=SEED
        )
    meta_lr.fit(meta_X[tr_idx], meta_y[tr_idx])
 
    fold_probs = meta_lr.predict_proba(meta_X[vl_idx])
    fold_preds = fold_probs.argmax(axis=1)
    stack_oof_probs[vl_idx] = fold_probs
    stack_oof_preds[vl_idx] = fold_preds
 
    fold_f1 = f1_score(meta_y[vl_idx], fold_preds, average='macro')
    stack_fold_scores.append(fold_f1)
 
    stack_test_probs += meta_lr.predict_proba(meta_X_test) / 5
    print(f' Fold {fold+1}: F1={fold_f1:.4f}')

stack_oof_f1 = f1_score(meta_y, stack_oof_preds, average='macro')
stack_cv_mean = np.mean(stack_fold_scores)
stack_cv_std = np.std(stack_fold_scores)

print(f'\nStacking Meta-Learner Results:')
print(f' CV Mean F1: {stack_cv_mean:.4f} ± {stack_cv_std:.4f}')
print(f' OOF F1: {stack_oof_f1:.4f}')

# Also train a final meta-learner on ALL OOF data for test predictions
final_meta_lr = LogisticRegression(
    C=1.0, class_weight='balanced', max_iter=5000,
    solver='lbfgs', random_state=SEED
    )
final_meta_lr.fit(meta_X, meta_y)
stack_test_probs_final = final_meta_lr.predict_proba(meta_X_test)
stack_test_preds = stack_test_probs_final.argmax(axis=1)

print(f'\nTest prediction distribution (stacking):')
for label in range(NUM_LABELS):
    print(f' {LABEL_MAP[label]:10s}: {(stack_test_preds == label).sum()}')

## 4. Method B -- Rank Averaging

**Problem with probability averaging:** Different models produce probabilities on different scales. Even after calibration, a DeBERTa "0.9" might mean something different from an SVC "0.9".

**Rank averaging** solves this by:
1. For each model, rank all test samples by their probability for each class (0 to N-1)
2. Average the ranks instead of the raw probabilities
3. Predict the class with the highest average rank

This is **robust to miscalibration** because it only cares about the relative ordering.

In [ ]:
def rank_average(prob_list, weights=None):
    """Rank-based averaging of probability matrices.
 
    For each model and each class, convert probabilities to ranks (0 to N-1),
    then compute weighted average of ranks.
    """
    n_models = len(prob_list)
    n_samples, n_classes = prob_list[0].shape
 
    if weights is None:
        weights = np.ones(n_models) / n_models
        weights = np.array(weights) / np.array(weights).sum()
 
        rank_sum = np.zeros((n_samples, n_classes))
 
        for model_idx, probs in enumerate(prob_list):
            for cls in range(n_classes):
                # Convert probabilities to ranks (higher prob → higher rank)
                ranks = probs[:, cls].argsort().argsort().astype(float)
                rank_sum[:, cls] += weights[model_idx] * ranks
 
                return rank_sum


                # Rank averaging on test set with Phase 4 weights
w = ensemble_config['ensemble_weights']
rank_weights = [w['deberta'], w['svc'], w['lr']]

test_prob_list = [deberta_test_probs, svc_test_probs_cv, lr_test_probs_cv]
rank_scores_test = rank_average(test_prob_list, weights=rank_weights)
rank_test_preds = rank_scores_test.argmax(axis=1)

# Also compute rank averaging on OOF (val+cal only for DeBERTa)
oof_prob_list = [
    deberta_oof_probs[oof_indices],
    svc_oof_probs[oof_indices],
    lr_oof_probs[oof_indices],
    ]
rank_scores_oof = rank_average(oof_prob_list, weights=rank_weights)
rank_oof_preds = rank_scores_oof.argmax(axis=1)
rank_oof_f1 = f1_score(meta_y, rank_oof_preds, average='macro')

print(f'Rank Averaging OOF F1: {rank_oof_f1:.4f}')
print(f'\nTest prediction distribution (rank averaging):')
for label in range(NUM_LABELS):
    print(f' {LABEL_MAP[label]:10s}: {(rank_test_preds == label).sum()}')

## 5. Method C -- Per-Class Specialist Voting

**Insight from Phase 4 error analysis:** Different models excel at different classes. For example:
- DeBERTa might be best at DeepSeek and Claude (rare classes)
- SVC might be best at Human (majority class)

**Per-class specialist voting** assigns each class to the model that performs best on it (measured by OOF per-class F1), then uses only that model's prediction for each class.

In [ ]:
# Compute per-class F1 for each model on OOF data
model_names = ['DeBERTa', 'SVC', 'LR']
oof_prob_arrays = [
    deberta_oof_probs[oof_indices],
    svc_oof_probs[oof_indices],
    lr_oof_probs[oof_indices],
    ]

per_class_f1 = np.zeros((3, NUM_LABELS))
for m_idx, probs in enumerate(oof_prob_arrays):
    preds = probs.argmax(axis=1)
    per_class_f1[m_idx] = f1_score(meta_y, preds, average=None)

    # Display per-class F1 table
print('Per-class F1 by model (on OOF data):')
print(f'{"":12s}', end='')
for cls in range(NUM_LABELS):
    print(f'{LABEL_MAP[cls]:>10s}', end='')
print()
for m_idx, name in enumerate(model_names):
    print(f'{name:12s}', end='')
    for cls in range(NUM_LABELS):
        marker = ' ' if per_class_f1[m_idx, cls] == per_class_f1[:, cls].max() else ' '
        print(f'{per_class_f1[m_idx, cls]:7.4f}{marker}', end='')
        print()

        # Assign each class to its best model
best_model_per_class = per_class_f1.argmax(axis=0)
print(f'\nClass specialists:')
for cls in range(NUM_LABELS):
    print(f' {LABEL_MAP[cls]:10s} → {model_names[best_model_per_class[cls]]}')

    # Build specialist predictions
    # For each test sample, use the specialist model's probability to pick the class
    # This is a soft version: boost the probability of the specialist's predicted class
test_probs_list = [deberta_test_probs, svc_test_probs_cv, lr_test_probs_cv]

specialist_test_probs = np.zeros((len(test_df), NUM_LABELS))
for cls in range(NUM_LABELS):
    best_m = best_model_per_class[cls]
    specialist_test_probs[:, cls] = test_probs_list[best_m][:, cls]

specialist_test_preds = specialist_test_probs.argmax(axis=1)

# OOF evaluation
specialist_oof_probs = np.zeros((len(oof_indices), NUM_LABELS))
for cls in range(NUM_LABELS):
    best_m = best_model_per_class[cls]
    specialist_oof_probs[:, cls] = oof_prob_arrays[best_m][:, cls]

specialist_oof_preds = specialist_oof_probs.argmax(axis=1)
specialist_oof_f1 = f1_score(meta_y, specialist_oof_preds, average='macro')

print(f'\nSpecialist OOF F1: {specialist_oof_f1:.4f}')
print(f'\nTest prediction distribution (specialist):')
for label in range(NUM_LABELS):
    print(f' {LABEL_MAP[label]:10s}: {(specialist_test_preds == label).sum()}')

## 6. Method D -- Confidence-Aware Blending

**Idea:** Instead of using the same weights for every sample, adjust weights per-sample based on each model's confidence.

If DeBERTa is very confident (max probability ≥ 0.95) for a sample, trust it more. If SVC is uncertain, trust it less.

**Formula:** For sample $i$ and model $m$:
$$w_{i,m} = \text{base\_weight}_m \times \text{confidence}_{i,m}^\alpha$$

where confidence = max probability and $\alpha$ controls how much we boost confident models.

In [ ]:
def confidence_blend(prob_list, base_weights, alpha=2.0):
    """Blend probabilities with per-sample confidence weighting.
 
    Models that are more confident on a given sample get more weight.
    alpha controls the sharpness (higher alpha = more weight to confident models).
    """
    n_samples = prob_list[0].shape[0]
    n_classes = prob_list[0].shape[1]
    n_models = len(prob_list)
 
    base_weights = np.array(base_weights)
    blended = np.zeros((n_samples, n_classes))
 
    for i in range(n_samples):
        # Per-model confidence for this sample
        confidences = np.array([probs[i].max() for probs in prob_list])
        # Adjusted weights = base_weight × confidence^alpha
        adjusted = base_weights * (confidences ** alpha)
        adjusted = adjusted / adjusted.sum() # normalize
 
        for m in range(n_models):
            blended[i] += adjusted[m] * prob_list[m][i]
 
            return blended


            # Search for best alpha on OOF data
w = ensemble_config['ensemble_weights']
base_w = [w['deberta'], w['svc'], w['lr']]

print('Searching for optimal alpha on OOF data...')
best_alpha = 0.0
best_conf_f1 = 0.0

for alpha in np.arange(0.0, 5.1, 0.5):
    blended = confidence_blend(oof_prob_arrays, base_w, alpha=alpha)
    preds = blended.argmax(axis=1)
    f1 = f1_score(meta_y, preds, average='macro')
    marker = ' ' if f1 > best_conf_f1 else ''
    print(f' alpha={alpha:.1f}: F1={f1:.4f}{marker}')
    if f1 > best_conf_f1:
        best_conf_f1 = f1
        best_alpha = alpha

print(f'\nBest alpha: {best_alpha:.1f} → OOF F1: {best_conf_f1:.4f}')

# Apply to test
conf_test_probs = confidence_blend(test_probs_list, base_w, alpha=best_alpha)
conf_test_preds = conf_test_probs.argmax(axis=1)

print(f'\nTest prediction distribution (confidence-aware):')
for label in range(NUM_LABELS):
    print(f' {LABEL_MAP[label]:10s}: {(conf_test_preds == label).sum()}')

## 7. Method E -- Threshold-Optimized Stacking

Take the best-performing method from above and apply per-class threshold optimization on top of it. This is a **two-stage** approach:
1. Stacking or blending produces calibrated ensemble probabilities
2. Per-class multipliers adjust the decision boundary for minority classes

We optimize thresholds on a held-out portion of the OOF data.

In [ ]:
def optimize_thresholds(probs, labels, n_classes=6, steps=50):
    """Greedy per-class threshold multiplier optimization.
 
    Instead of argmax(probs), use argmax(probs × multipliers)
    where multiplier > 1 boosts a class, < 1 suppresses it.
    """
    best_multipliers = np.ones(n_classes)
    best_f1 = f1_score(labels, probs.argmax(axis=1), average='macro')
 
    for iteration in range(3):
        improved = False
        for cls in range(n_classes):
            for mult in np.linspace(0.5, 2.0, steps):
                trial = best_multipliers.copy()
                trial[cls] = mult
                preds = (probs * trial).argmax(axis=1)
                f1 = f1_score(labels, preds, average='macro')
                if f1 > best_f1 + 1e-6:
                    best_f1 = f1
                    best_multipliers = trial.copy()
                    improved = True
                    if not improved:
                        break
 
                        return best_multipliers, best_f1


                        # Apply threshold optimization to the stacking probabilities
print('Optimizing thresholds on stacking OOF probabilities...')
stack_thresholds, stack_thresh_f1 = optimize_thresholds(stack_oof_probs, meta_y)

print(f'\nStacking + Thresholds OOF F1: {stack_thresh_f1:.4f} (was {stack_oof_f1:.4f})')
print(f'Multipliers:')
for cls in range(NUM_LABELS):
    print(f' {LABEL_MAP[cls]:10s}: {stack_thresholds[cls]:.3f}')

stack_thresh_test_preds = (stack_test_probs_final * stack_thresholds).argmax(axis=1)

print(f'\nTest prediction distribution (stacking + thresholds):')
for label in range(NUM_LABELS):
    print(f' {LABEL_MAP[label]:10s}: {(stack_thresh_test_preds == label).sum()}')

## 8. Grand Comparison -- All Methods

Now we compare every ensemble method side-by-side. This includes:
- **Phase 4 baseline** (weighted soft voting from the previous notebook)
- **Stacking** (meta-learner)
- **Rank averaging** (rank-based blend)
- **Specialist voting** (per-class best model)
- **Confidence-aware blending** (dynamic weights)
- **Stacking + thresholds** (two-stage optimization)

The bar chart makes it easy to see which approach is best.

In [ ]:
# Also compute Phase 4 soft-voting OOF F1 for comparison
w = ensemble_config['ensemble_weights']
phase4_oof_probs = (
    w['deberta'] * deberta_oof_probs[oof_indices] +
    w['svc'] * svc_oof_probs[oof_indices] +
    w['lr'] * lr_oof_probs[oof_indices]
    )
phase4_oof_f1 = f1_score(meta_y, phase4_oof_probs.argmax(axis=1), average='macro')

# Collect all results
all_methods = [
    ('Phase 4: Soft Voting', phase4_oof_f1),
    ('Stacking (Meta-LR)', stack_oof_f1),
    ('Rank Averaging', rank_oof_f1),
    ('Specialist Voting', specialist_oof_f1),
    ('Confidence Blending', best_conf_f1),
    ('Stacking + Thresholds', stack_thresh_f1),
    ]

# Sort by F1
all_methods.sort(key=lambda x: x[1], reverse=True)

print('=' * 60)
print('GRAND COMPARISON -- ALL ENSEMBLE METHODS')
print('=' * 60)
print(f'{"Method":<35} {"OOF F1":>10}')
print('-' * 50)
for name, f1 in all_methods:
    marker = ' BEST' if f1 == all_methods[0][1] else ''
    print(f'{name:<35} {f1:>10.4f}{marker}')

    # --- Bar chart ---
fig, ax = plt.subplots(figsize=(12, 6))

names = [m[0] for m in all_methods]
scores = [m[1] for m in all_methods]
colors = ['#2ecc71' if i == 0 else '#3498db' for i in range(len(all_methods))]

bars = ax.barh(range(len(names)), scores, color=colors, edgecolor='black', linewidth=0.5)
ax.set_yticks(range(len(names)))
ax.set_yticklabels(names)
ax.set_xlabel('OOF Macro F1')
ax.set_title('Ensemble Method Comparison -- OOF Macro F1', fontweight='bold', fontsize=14)

# Add value labels on bars
for bar, score in zip(bars, scores):
    ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
        f'{score:.4f}', va='center', fontweight='bold')

    # Set x-axis to show differences clearly
min_score = min(scores) - 0.01
max_score = max(scores) + 0.01
ax.set_xlim(min_score, max_score)

ax.invert_yaxis()
plt.tight_layout()
plt.savefig('figures/advanced_ensemble_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: figures/advanced_ensemble_comparison.png')

## 9. Per-Class F1 Breakdown

Let's see which method does best for each class, especially the hard minority classes (DeepSeek, Claude).

In [ ]:
# Per-class F1 for each method
method_preds = {
    'Phase 4 Soft Vote': phase4_oof_probs.argmax(axis=1),
    'Stacking': stack_oof_preds,
    'Rank Average': rank_oof_preds,
    'Specialist': specialist_oof_preds,
    'Confidence Blend': confidence_blend(oof_prob_arrays, base_w, alpha=best_alpha).argmax(axis=1),
    'Stack+Thresh': (stack_oof_probs * stack_thresholds).argmax(axis=1),
    }

per_class_df = pd.DataFrame(index=[LABEL_MAP[i] for i in range(NUM_LABELS)])
for method_name, preds in method_preds.items():
    per_class_df[method_name] = f1_score(meta_y, preds, average=None)

per_class_df['Best Method'] = per_class_df.drop(columns='Best Method', errors='ignore').idxmax(axis=1)
print('Per-Class F1 by Method:')
print(per_class_df.to_string(float_format='{:.4f}'.format))

# Bar chart of per-class F1
fig, ax = plt.subplots(figsize=(14, 6))
plot_df = per_class_df.drop(columns='Best Method')
plot_df.plot(kind='bar', ax=ax, width=0.8, edgecolor='black', linewidth=0.5)
ax.set_ylabel('F1 Score')
ax.set_title('Per-Class F1 by Ensemble Method', fontweight='bold', fontsize=14)
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)
ax.set_xticklabels([LABEL_MAP[i] for i in range(NUM_LABELS)], rotation=30)
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.savefig('figures/advanced_ensemble_per_class.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: figures/advanced_ensemble_per_class.png')

## 10. Select Best Method & Save Everything

Based on the comparison above, we pick the best overall method and generate final submissions. We also save all probabilities so the Final Polish notebook (Phase 6) can do the ultimate selection.

In [ ]:
# Identify the best method
best_method_name, best_method_f1 = all_methods[0]
print(f'Best method: {best_method_name} (OOF F1: {best_method_f1:.4f})')

# Map method names to test predictions and probabilities
method_test_preds = {
    'Phase 4: Soft Voting': (w['deberta'] * deberta_test_probs + w['svc'] * svc_test_probs_cv + w['lr'] * lr_test_probs_cv).argmax(axis=1),
    'Stacking (Meta-LR)': stack_test_preds,
    'Rank Averaging': rank_test_preds,
    'Specialist Voting': specialist_test_preds,
    'Confidence Blending': conf_test_preds,
    'Stacking + Thresholds': stack_thresh_test_preds,
    }

method_test_probs = {
    'Phase 4: Soft Voting': w['deberta'] * deberta_test_probs + w['svc'] * svc_test_probs_cv + w['lr'] * lr_test_probs_cv,
    'Stacking (Meta-LR)': stack_test_probs_final,
    'Rank Averaging': rank_scores_test,
    'Specialist Voting': specialist_test_probs,
    'Confidence Blending': conf_test_probs,
    'Stacking + Thresholds': stack_test_probs_final, # thresholds applied at prediction time
    }

### 💾 Save Submissions & Stacking Model

Write predictions for every ensemble method to CSV and persist the stacking meta-learner for Phase 6.

In [ ]:
# Save submissions for each method
os.makedirs('submissions', exist_ok=True)
os.makedirs('models', exist_ok=True)

short_names = {
    'Phase 4: Soft Voting': 'phase4_softvote',
    'Stacking (Meta-LR)': 'phase5_stacking',
    'Rank Averaging': 'phase5_rank_avg',
    'Specialist Voting': 'phase5_specialist',
    'Confidence Blending': 'phase5_conf_blend',
    'Stacking + Thresholds': 'phase5_stack_thresh',
    }

print('Saving submissions...')
for method_name, preds in method_test_preds.items():
    short = short_names[method_name]
    path = f'submissions/{short}.csv'
    lines = ['ID,LABEL'] + [f'{i},{preds[i]}' for i in range(600)]
    with open(path, 'w') as f:
        f.write('\n'.join(lines) + '\n')
        dist = dict(zip(*np.unique(preds, return_counts=True)))
        print(f' {short:30s} dist={dist}')

        # Save stacking model
joblib.dump(final_meta_lr, 'models/stacking_meta_lr.joblib')
print('\nSaved: models/stacking_meta_lr.joblib')

# Save all test probabilities for Phase 6 final selection
np.save('models/stacking_test_probs.npy', stack_test_probs_final)
np.save('models/stacking_conf_blend_test_probs.npy', conf_test_probs)
np.save('models/stacking_rank_scores_test.npy', rank_scores_test)
np.save('models/stacking_specialist_test_probs.npy', specialist_test_probs)

# Save config
stacking_config = {
    'best_method': best_method_name,
    'best_oof_f1': float(best_method_f1),
    'stacking_cv_f1': f'{stack_cv_mean:.4f} ± {stack_cv_std:.4f}',
    'rank_avg_oof_f1': float(rank_oof_f1),
    'specialist_oof_f1': float(specialist_oof_f1),
    'confidence_blend_alpha': float(best_alpha),
    'confidence_blend_oof_f1': float(best_conf_f1),
    'stack_thresholds': stack_thresholds.tolist(),
    'stack_thresh_oof_f1': float(stack_thresh_f1),
    'best_model_per_class': {LABEL_MAP[i]: model_names[best_model_per_class[i]] for i in range(NUM_LABELS)},
    'all_methods_ranked': [(name, float(f1)) for name, f1 in all_methods],
    }

with open('models/stacking_config.json', 'w') as f:
    json.dump(stacking_config, f, indent=2)

print('\nSaved: models/stacking_config.json')
print('\n All Phase 5 artifacts saved!')

### 💾 Save Submissions & Models

Write predictions for every ensemble method to CSV and persist the stacking meta-learner.

## 11. Phase 5 Summary

Run this cell to see a complete summary of everything we tried and learned.

In [ ]:
print('=' * 70)
print('PHASE 5 SUMMARY -- ADVANCED ENSEMBLE')
print('=' * 70)

print('\nBase Models:')
print(f' DeBERTa-v3-base val F1: {deberta_val_f1:.4f}')
print(f' SVC (C=5.0) 5-fold OOF: {svc_oof_f1:.4f}')
print(f' LR (C=2.0) 5-fold OOF: {lr_oof_f1:.4f}')

print('\nAdvanced Ensemble Methods (OOF F1):')
for name, f1 in all_methods:
    marker = ' ← BEST' if f1 == all_methods[0][1] else ''
    print(f' {name:35s} {f1:.4f}{marker}')

print(f'\nClass Specialists:')
for cls in range(NUM_LABELS):
    print(f' {LABEL_MAP[cls]:10s} → {model_names[best_model_per_class[cls]]}')

print(f'\nConfidence Blending alpha: {best_alpha:.1f}')
print(f'Stacking threshold multipliers: {dict(zip([LABEL_MAP[i] for i in range(NUM_LABELS)], [f"{t:.3f}" for t in stack_thresholds]))}')

print(f'\nSubmissions saved: {len(method_test_preds)} files in submissions/')
print(f'Artifacts saved in models/ for Phase 6 final selection.')
print('\n Phase 5 complete! Proceed to Phase 6 for final polish.')